In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

print("Feature engineering environment ready!")

Feature engineering environment ready!


In [2]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Input and output directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"

# Create feature directory
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Input file
INPUT_FILE = PROCESSED_DIR / "battery_discharge_cycles.csv"

print("Project root:")
print(PROJECT_ROOT)

print("\nInput file:")
print(INPUT_FILE)

print("\nInput file exists:", INPUT_FILE.exists())

Project root:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ

Input file:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\processed\battery_discharge_cycles.csv

Input file exists: True


In [3]:
# Load processed dataset

df = pd.read_csv(INPUT_FILE)

print("Processed dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Processed dataset loaded successfully!
Shape: (185721, 9)

Columns:
['battery_id', 'cycle', 'time', 'voltage', 'current', 'temperature', 'current_load', 'voltage_load', 'capacity']

First 5 rows:


,battery_id,cycle,time,voltage,current,temperature,current_load,voltage_load,capacity
0,B0005,1,0.000,4.191492,-0.004902,24.330034,-0.0006,0.000,1.856487
1,B0005,1,16.781,4.190749,-0.001478,24.325993,-0.0006,4.206,1.856487
2,B0005,1,35.703,3.974871,-2.012528,24.389085,-1.9982,3.062,1.856487
3,B0005,1,53.781,3.951717,-2.013979,24.544752,-1.9982,3.030,1.856487
4,B0005,1,71.922,3.934352,-2.011144,24.731385,-1.9982,3.011,1.856487


In [4]:
# Validate processed dataset before feature engineering

print("Dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of unique batteries:")
print(df["battery_id"].nunique())

print("\nBatteries:")
print(df["battery_id"].unique())

print("\nCycle range:")
print(df["cycle"].min(), "to", df["cycle"].max())

Dataset shape: (185721, 9)

Data types:
battery_id       object
cycle             int64
time            float64
voltage         float64
current         float64
temperature     float64
current_load    float64
voltage_load    float64
capacity        float64
dtype: object

Missing values:
battery_id      0
cycle           0
time            0
voltage         0
current         0
temperature     0
current_load    0
voltage_load    0
capacity        0
dtype: int64

Number of unique batteries:
4

Batteries:
['B0005' 'B0006' 'B0007' 'B0018']

Cycle range:
1 to 168


In [5]:
# Create cycle-level features

cycle_features = (
    df.groupby(["battery_id", "cycle"])
    .agg(
        voltage_mean=("voltage", "mean"),
        voltage_min=("voltage", "min"),
        voltage_max=("voltage", "max"),
        voltage_std=("voltage", "std"),

        current_mean=("current", "mean"),
        current_min=("current", "min"),
        current_max=("current", "max"),

        temperature_mean=("temperature", "mean"),
        temperature_max=("temperature", "max"),

        current_load_mean=("current_load", "mean"),
        voltage_load_mean=("voltage_load", "mean"),

        capacity=("capacity", "first"),
        cycle_time=("time", "max"),
    )
    .reset_index()
)

# Replace NaN standard deviation values for single-row groups
cycle_features["voltage_std"] = cycle_features["voltage_std"].fillna(0)

print("Cycle-level features created successfully!")
print("Shape:", cycle_features.shape)

display(cycle_features.head())

Cycle-level features created successfully!
Shape: (636, 15)


,battery_id,cycle,voltage_mean,voltage_min,voltage_max,voltage_std,current_mean,current_min,current_max,temperature_mean,temperature_max,current_load_mean,voltage_load_mean,capacity,cycle_time
0,B0005,1,3.529829,2.612467,4.191492,0.236558,-1.818702,-2.018015,0.000729,32.572328,38.982181,-1.805570,2.404944,1.856487,3690.234
1,B0005,2,3.537320,2.587209,4.189773,0.235366,-1.817560,-2.016821,0.002927,32.725235,39.033398,-1.804583,2.399260,1.846327,3672.344
2,B0005,3,3.543737,2.651917,4.188187,0.228111,-1.816487,-2.016574,0.001484,32.642862,38.818797,-1.803575,2.397969,1.835349,3651.641
3,B0005,4,3.543666,2.592948,4.188461,0.233347,-1.825589,-2.015936,0.001547,32.514876,38.762305,-1.812863,2.408289,1.835263,3631.563
4,B0005,5,3.542343,2.547420,4.188299,0.237301,-1.826114,-2.017426,0.001701,32.382349,38.665393,-1.812876,2.408505,1.834646,3629.172


In [6]:
# Create degradation-related features

cycle_features = cycle_features.sort_values(
    ["battery_id", "cycle"]
).reset_index(drop=True)

# Capacity change from the previous cycle
cycle_features["capacity_change"] = (
    cycle_features.groupby("battery_id")["capacity"].diff()
)

# Capacity retention relative to the first cycle
cycle_features["capacity_retention"] = (
    cycle_features["capacity"]
    / cycle_features.groupby("battery_id")["capacity"].transform("first")
)

# Voltage change from the previous cycle
cycle_features["voltage_change"] = (
    cycle_features.groupby("battery_id")["voltage_mean"].diff()
)

# Temperature change from the previous cycle
cycle_features["temperature_change"] = (
    cycle_features.groupby("battery_id")["temperature_mean"].diff()
)

# Fill first-cycle differences with 0
difference_columns = [
    "capacity_change",
    "voltage_change",
    "temperature_change"
]

cycle_features[difference_columns] = (
    cycle_features[difference_columns].fillna(0)
)

print("Degradation features created successfully!")
print("Shape:", cycle_features.shape)

display(
    cycle_features[
        [
            "battery_id",
            "cycle",
            "capacity",
            "capacity_change",
            "capacity_retention",
            "voltage_change",
            "temperature_change"
        ]
    ].head(10)
)

Degradation features created successfully!


Shape: (636, 19)


,battery_id,cycle,capacity,capacity_change,capacity_retention,voltage_change,temperature_change
0,B0005,1,1.856487,0.000000,1.000000,0.000000,0.000000
1,B0005,2,1.846327,-0.010160,0.994527,0.007491,0.152907
2,B0005,3,1.835349,-0.010978,0.988614,0.006417,-0.082373
3,B0005,4,1.835263,-0.000087,0.988567,-0.000071,-0.127985
4,B0005,5,1.834646,-0.000617,0.988235,-0.001323,-0.132527
5,B0005,6,1.835662,0.001016,0.988782,-0.001009,0.051833
6,B0005,7,1.835146,-0.000516,0.988504,-0.000310,0.046234
7,B0005,8,1.825757,-0.009389,0.983447,0.013108,-0.069955
8,B0005,9,1.824774,-0.000983,0.982917,-0.001198,-0.064321
9,B0005,10,1.824613,-0.000161,0.982831,-0.001730,-0.069343


In [7]:
# Create additional battery degradation features

# Rolling average of capacity to smooth cycle-to-cycle noise
cycle_features["capacity_rolling_mean"] = (
    cycle_features.groupby("battery_id")["capacity"]
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

# Rolling average of temperature
cycle_features["temperature_rolling_mean"] = (
    cycle_features.groupby("battery_id")["temperature_mean"]
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

# Capacity degradation from the first cycle
cycle_features["capacity_degradation"] = (
    1 - cycle_features["capacity_retention"]
)

print("Additional degradation features created successfully!")
print("Shape:", cycle_features.shape)

display(
    cycle_features[
        [
            "battery_id",
            "cycle",
            "capacity",
            "capacity_retention",
            "capacity_degradation",
            "capacity_rolling_mean",
            "temperature_rolling_mean"
        ]
    ].head(10)
)


Additional degradation features created successfully!
Shape: (636, 22)


,battery_id,cycle,capacity,capacity_retention,capacity_degradation,capacity_rolling_mean,temperature_rolling_mean
0,B0005,1,1.856487,1.000000,0.000000,1.856487,32.572328
1,B0005,2,1.846327,0.994527,0.005473,1.851407,32.648782
2,B0005,3,1.835349,0.988614,0.011386,1.846055,32.646808
3,B0005,4,1.835263,0.988567,0.011433,1.843357,32.613825
4,B0005,5,1.834646,0.988235,0.011765,1.841614,32.567530
5,B0005,6,1.835662,0.988782,0.011218,1.837449,32.539901
6,B0005,7,1.835146,0.988504,0.011496,1.835213,32.490937
7,B0005,8,1.825757,0.983447,0.016553,1.833295,32.444457
8,B0005,9,1.824774,0.982917,0.017083,1.831197,32.410710
9,B0005,10,1.824613,0.982831,0.017169,1.829190,32.389600


In [8]:
# Select final feature columns

feature_columns = [
    "battery_id",
    "cycle",
    "voltage_mean",
    "voltage_min",
    "voltage_max",
    "voltage_std",
    "current_mean",
    "current_min",
    "current_max",
    "temperature_mean",
    "temperature_max",
    "current_load_mean",
    "voltage_load_mean",
    "cycle_time",
    "capacity",
    "capacity_change",
    "capacity_retention",
    "voltage_change",
    "temperature_change",
    "capacity_rolling_mean",
    "temperature_rolling_mean",
    "capacity_degradation"
]

features_df = cycle_features[feature_columns].copy()

print("Final feature dataset created!")
print("Shape:", features_df.shape)

print("\nColumns:")
print(features_df.columns.tolist())

display(features_df.head())

Final feature dataset created!
Shape: (636, 22)

Columns:
['battery_id', 'cycle', 'voltage_mean', 'voltage_min', 'voltage_max', 'voltage_std', 'current_mean', 'current_min', 'current_max', 'temperature_mean', 'temperature_max', 'current_load_mean', 'voltage_load_mean', 'cycle_time', 'capacity', 'capacity_change', 'capacity_retention', 'voltage_change', 'temperature_change', 'capacity_rolling_mean', 'temperature_rolling_mean', 'capacity_degradation']


,battery_id,cycle,voltage_mean,voltage_min,voltage_max,voltage_std,current_mean,current_min,current_max,temperature_mean,...,voltage_load_mean,cycle_time,capacity,capacity_change,capacity_retention,voltage_change,temperature_change,capacity_rolling_mean,temperature_rolling_mean,capacity_degradation
0,B0005,1,3.529829,2.612467,4.191492,0.236558,-1.818702,-2.018015,0.000729,32.572328,...,2.404944,3690.234,1.856487,0.000000,1.000000,0.000000,0.000000,1.856487,32.572328,0.000000
1,B0005,2,3.537320,2.587209,4.189773,0.235366,-1.817560,-2.016821,0.002927,32.725235,...,2.399260,3672.344,1.846327,-0.010160,0.994527,0.007491,0.152907,1.851407,32.648782,0.005473
2,B0005,3,3.543737,2.651917,4.188187,0.228111,-1.816487,-2.016574,0.001484,32.642862,...,2.397969,3651.641,1.835349,-0.010978,0.988614,0.006417,-0.082373,1.846055,32.646808,0.011386
3,B0005,4,3.543666,2.592948,4.188461,0.233347,-1.825589,-2.015936,0.001547,32.514876,...,2.408289,3631.563,1.835263,-0.000087,0.988567,-0.000071,-0.127985,1.843357,32.613825,0.011433
4,B0005,5,3.542343,2.547420,4.188299,0.237301,-1.826114,-2.017426,0.001701,32.382349,...,2.408505,3629.172,1.834646,-0.000617,0.988235,-0.001323,-0.132527,1.841614,32.567530,0.011765


In [9]:
# Separate target information from ML features
# SOH is calculated from measured discharge capacity.
#
# SOH (%) = Current Capacity / Rated Capacity × 100
# Rated capacity = 2.0 Ah

RATED_CAPACITY = 2.0  # Ah

TARGET_COLUMNS = [
    "battery_id",
    "cycle",
    "capacity"
]

targets_df = features_df[TARGET_COLUMNS].copy()

# Calculate State of Health (SOH)
targets_df["soh"] = (
    targets_df["capacity"] / RATED_CAPACITY
) * 100

print("Target dataset created!")
print("Shape:", targets_df.shape)

print("\nTarget columns:")
print(targets_df.columns.tolist())

print("\nSOH range:")
print(
    f"{targets_df['soh'].min():.2f}% - "
    f"{targets_df['soh'].max():.2f}%"
)

display(targets_df.head())

Target dataset created!
Shape: (636, 4)

Target columns:
['battery_id', 'cycle', 'capacity', 'soh']

SOH range:
57.69% - 101.77%


,battery_id,cycle,capacity,soh
0,B0005,1,1.856487,92.824371
1,B0005,2,1.846327,92.316362
2,B0005,3,1.835349,91.767460
3,B0005,4,1.835263,91.763126
4,B0005,5,1.834646,91.732275


In [10]:
# Remove target-derived columns from the ML feature dataset
# to prevent target leakage during SOH prediction.

LEAKAGE_COLUMNS = [
    "capacity",
    "capacity_change",
    "capacity_retention",
    "capacity_rolling_mean",
    "capacity_degradation"
]

features_df = features_df.drop(columns=LEAKAGE_COLUMNS)

print("Leakage-prone columns removed.")
print("Feature dataset shape:", features_df.shape)

print("\nRemaining features:")
print(features_df.columns.tolist())

Leakage-prone columns removed.
Feature dataset shape: (636, 17)

Remaining features:
['battery_id', 'cycle', 'voltage_mean', 'voltage_min', 'voltage_max', 'voltage_std', 'current_mean', 'current_min', 'current_max', 'temperature_mean', 'temperature_max', 'current_load_mean', 'voltage_load_mean', 'cycle_time', 'voltage_change', 'temperature_change', 'temperature_rolling_mean']


In [11]:
# Save engineered features

OUTPUT_FILE = FEATURE_DIR / "battery_cycle_features.csv"

features_df.to_csv(OUTPUT_FILE, index=False)

# Save target dataset separately
TARGET_OUTPUT_FILE = FEATURE_DIR / "battery_cycle_targets.csv"

targets_df.to_csv(TARGET_OUTPUT_FILE, index=False)

print("\nTarget dataset saved successfully!")
print("File:")
print(TARGET_OUTPUT_FILE)

print("\nTarget dataset shape:", targets_df.shape)
print(
    "File size:",
    round(TARGET_OUTPUT_FILE.stat().st_size / (1024 * 1024), 2),
    "MB"
)
print("Feature dataset saved successfully!")
print("\nFile:")
print(OUTPUT_FILE)

print("\nShape:", features_df.shape)
print(
    "File size:",
    round(OUTPUT_FILE.stat().st_size / (1024 * 1024), 2),
    "MB"
)


Target dataset saved successfully!
File:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\features\battery_cycle_targets.csv

Target dataset shape: (636, 4)
File size: 0.03 MB
Feature dataset saved successfully!

File:
d:\VIT Academic\TY\sem 5\EDI\BatteryIQ\data\features\battery_cycle_features.csv

Shape: (636, 17)
File size: 0.17 MB


In [12]:
# Verify saved feature dataset

features_check = pd.read_csv(OUTPUT_FILE)
targets_check = pd.read_csv(TARGET_OUTPUT_FILE)

print("=== FEATURE DATASET VALIDATION ===")

print("Feature shape:", features_check.shape)

print("\nFeature columns:")
print(features_check.columns.tolist())

print("\nMissing values:")
print(features_check.isna().sum().sum())

print("\nDuplicate rows:")
print(features_check.duplicated().sum())

print("\nBatteries:")
print(features_check["battery_id"].unique())

print("\nCycles per battery:")
print(
    features_check.groupby("battery_id")["cycle"].nunique()
)

print("=== TARGET DATASET VALIDATION ===")

print("Target shape:", targets_check.shape)

print("\nTarget columns:")
print(targets_check.columns.tolist())

print("\nMissing values:")
print(targets_check.isna().sum())

print("\nDuplicate rows:")
print(targets_check.duplicated().sum())

print("\nBatteries:")
print(targets_check["battery_id"].unique())

print("\nCycles per battery:")
print(
    targets_check.groupby("battery_id")["cycle"].nunique()
)

print("\nCapacity range:")
print(
    f"{targets_check['capacity'].min():.4f} - "
    f"{targets_check['capacity'].max():.4f} Ah"
)

print("\nSOH range:")
print(
    f"{targets_check['soh'].min():.2f}% - "
    f"{targets_check['soh'].max():.2f}%"
)

print("\nSOH summary:")
display(targets_check["soh"].describe())

print("\n=== SOH CONSISTENCY CHECK ===")

expected_soh = (
    targets_check["capacity"] / RATED_CAPACITY
) * 100

soh_difference = (
    targets_check["soh"] - expected_soh
).abs()

print(
    "Maximum SOH calculation difference:",
    soh_difference.max()
)

print("\n=== LEAKAGE CHECK ===")

leakage_present = [
    column for column in LEAKAGE_COLUMNS
    if column in features_check.columns
]

print("Leakage-prone columns present in ML features:", leakage_present)

=== FEATURE DATASET VALIDATION ===
Feature shape: (636, 17)

Feature columns:
['battery_id', 'cycle', 'voltage_mean', 'voltage_min', 'voltage_max', 'voltage_std', 'current_mean', 'current_min', 'current_max', 'temperature_mean', 'temperature_max', 'current_load_mean', 'voltage_load_mean', 'cycle_time', 'voltage_change', 'temperature_change', 'temperature_rolling_mean']

Missing values:
0

Duplicate rows:
0

Batteries:
['B0005' 'B0006' 'B0007' 'B0018']

Cycles per battery:
battery_id
B0005    168
B0006    168
B0007    168
B0018    132
Name: cycle, dtype: int64
=== TARGET DATASET VALIDATION ===
Target shape: (636, 4)

Target columns:
['battery_id', 'cycle', 'capacity', 'soh']

Missing values:
battery_id    0
cycle         0
capacity      0
soh           0
dtype: int64

Duplicate rows:
0

Batteries:
['B0005' 'B0006' 'B0007' 'B0018']

Cycles per battery:
battery_id
B0005    168
B0006    168
B0007    168
B0018    132
Name: cycle, dtype: int64

Capacity range:
1.1538 - 2.0353 Ah

SOH range:


count    636.000000
mean      79.082591
std        9.938233
min       57.690917
25%       71.056133
50%       77.984736
75%       88.174302
max      101.766880
Name: soh, dtype: float64


=== SOH CONSISTENCY CHECK ===
Maximum SOH calculation difference: 4.263256414560601e-14

=== LEAKAGE CHECK ===
Leakage-prone columns present in ML features: []


In [13]:
# Final target integrity checks

assert targets_check.shape == (636, 4)

assert set(targets_check.columns) == {
    "battery_id",
    "cycle",
    "capacity",
    "soh"
}

assert targets_check.isna().sum().sum() == 0

assert targets_check.duplicated().sum() == 0

assert targets_check["capacity"].gt(0).all()

assert targets_check["soh"].gt(0).all()

assert soh_difference.max() < 1e-10

assert set(targets_check["battery_id"].unique()) == {
    "B0005", "B0006", "B0007", "B0018"
}

print("All target integrity checks passed!")

All target integrity checks passed!


In [14]:
# Final feature engineering checks

assert features_check.shape[0] == 636
assert targets_check.shape[0] == 636

assert features_check.isna().sum().sum() == 0
assert targets_check.isna().sum().sum() == 0

assert features_check.duplicated().sum() == 0
assert targets_check.duplicated().sum() == 0

assert not any(
    column in features_check.columns
    for column in LEAKAGE_COLUMNS
)

assert set(features_check["battery_id"].unique()) == {
    "B0005", "B0006", "B0007", "B0018"
}

print("All feature engineering validation checks passed!")

All feature engineering validation checks passed!


# Final ML Feature and Target Contract

## Predictor Features

The following variables are retained as ML predictors because they describe battery electrical, thermal, and discharge behavior without directly containing the target capacity:

### Voltage Features
- `voltage_mean`
- `voltage_min`
- `voltage_max`
- `voltage_std`
- `voltage_change`

### Current Features
- `current_mean`
- `current_min`
- `current_max`
- `current_load_mean`

### Temperature Features
- `temperature_mean`
- `temperature_max`
- `temperature_change`
- `temperature_rolling_mean`

### Discharge Behavior
- `voltage_load_mean`
- `cycle_time`

### Metadata
- `battery_id`
- `cycle`

`battery_id` and `cycle` are retained for battery-level grouping, temporal ordering, and leave-one-battery-out validation. They should not be treated as ordinary predictive measurements by the ML models.

## Target Variables

The target dataset contains:

- `capacity` — measured discharge capacity in Ah
- `soh` — State of Health (%)

SOH is calculated as:

**SOH = (Current Capacity / Rated Capacity) × 100**

where the rated capacity is 2.0 Ah.

## Target Leakage Policy

The following capacity-derived variables are excluded from the ML predictor dataset:

- `capacity`
- `capacity_change`
- `capacity_retention`
- `capacity_rolling_mean`
- `capacity_degradation`

These variables are directly derived from capacity, which is used to calculate SOH. Including them as predictors would introduce target leakage and could produce artificially optimistic model performance.

The final ML pipeline will therefore use only non-capacity-derived electrical, thermal, discharge, and temporal characteristics to predict battery SOH.

In [15]:
print("=== FINAL FEATURE/TARGET CONTRACT ===")

print("\nML feature columns:")
for column in features_check.columns:
    print(" -", column)

print("\nTarget columns:")
for column in targets_check.columns:
    print(" -", column)

print("\nFeature count:", features_check.shape[1])
print("Target count:", targets_check.shape[1])

print("\nMember 3 feature engineering pipeline is ML-ready.")

=== FINAL FEATURE/TARGET CONTRACT ===

ML feature columns:
 - battery_id
 - cycle
 - voltage_mean
 - voltage_min
 - voltage_max
 - voltage_std
 - current_mean
 - current_min
 - current_max
 - temperature_mean
 - temperature_max
 - current_load_mean
 - voltage_load_mean
 - cycle_time
 - voltage_change
 - temperature_change
 - temperature_rolling_mean

Target columns:
 - battery_id
 - cycle
 - capacity
 - soh

Feature count: 17
Target count: 4

Member 3 feature engineering pipeline is ML-ready.
